In [220]:
import time
from collections import defaultdict
from typing import Callable, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as sps
import seaborn as sns
import torch
from catboost import CatBoostRegressor
from pylab import rcParams
from sktime.split import ExpandingWindowSplitter
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
)
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import MinMaxScaler
from statsforecast import models as sf_models
from statsforecast.models import (
    SimpleExponentialSmoothing,
    SimpleExponentialSmoothingOptimized,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm
from tqdm.auto import tqdm as auto_tqdm

from IPython.display import HTML, clear_output, display
import plotly.graph_objects as go
import statsmodels
import statsmodels.api as sm

from statsmodels.tsa.holtwinters import ExponentialSmoothing, Holt, SimpleExpSmoothing
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
import tsfresh
from tsfresh import extract_features, select_features
from tsfresh.feature_extraction import settings
from tsfresh.utilities.dataframe_functions import impute, roll_time_series

# Visualization setup
sns.set(font_scale=1.3, palette="Set2")
rcParams['figure.figsize'] = 15, 7
%matplotlib inline

# Обработка

In [221]:
chemicals_raw = pd.read_excel(
    "/home/danilach/mipt-stats/SberTs/Цены удобрения и химия_до 06.2025-1.xlsx"
)

In [222]:
chemicals_raw = pd.read_excel(
    "/home/danilach/mipt-stats/SberTs/Цены удобрения и химия_до 06.2025-1.xlsx"
)

chemicals_raw = chemicals_raw.T
chemicals_raw.set_index(1)
chemicals_raw.columns = chemicals_raw.iloc[0]
chemicals_raw = chemicals_raw[2:]
chemicals_raw.index = pd.to_datetime(chemicals_raw.index)
chemicals_raw.index.name = "Date"
chemicals_raw.columns.name = "chemical"

In [223]:
products = chemicals_raw.dropna(axis="columns")
products.head()

chemical,Карбамид (FOB Южный),"Моноаммонийфосфат, MAP (FOB Балтика)",Апатитовый концетрат (FOB Morocco),Аммиак (FOB Черное море),Аммиачная селитра (FOB Черное море),Хлорид калия (CFR Ю-В Азия),Капролактам импортный контракт (Тайвань и Ю. Корея) CFR Азия,Метанол,"Бензол, CFR Япония","Этилен, CFR Китай"
Date,,,,,,,,,,
2016-01-01,209.875,366.25,120.0,266.666667,199.375,278.75,1153.0,185.4,556.0,987.5
2016-02-01,207.125,332.5,117.5,260.5,182.125,272.5,1163.0,150.0,546.25,927.5
2016-03-01,199.875,345.0,117.5,267.5,180.3,267.5,1290.0,164.75,625.0,1173.75
2016-04-01,203.875,346.25,115.0,271.0,172.0,267.5,1355.0,179.9,642.0,1192.5
2016-05-01,200.5,341.5,107.5,279.25,154.75,251.875,1278.0,201.25,631.25,1160.0


In [224]:
macro = pd.read_excel("/home/danilach/mipt-stats/SberTs/global_macro.xlsx", index_col="Date")   
macro.head()

,"Инфляция - Рост индекса цен производителей (RUB, eop PPI),","Инфляция - Рост индекса потребительских цен в США, в долларах США (USD, eop CPI),","Инфляция - Рост индекса потребительских цен в США, в долларах США (USD, eop CPI), .1","Инфляция - Рост индекса цен производителей в США, в долларах США (USD, eop PPI),","Ключевая ставка, годовых","Инфляция, г/г"
Date,,,,,,
"08,2025",NaN,0.2,0.2,0.009,NaN,NaN
"07,2025",-0.013,0.3,0.3,0.000,18.0,8.79
"06,2025",-0.013,0.1,0.1,0.001,20.0,9.40
"05,2025",-0.014,0.2,0.2,-0.005,21.0,9.88
"04,2025",-0.015,-0.1,-0.1,-0.004,21.0,10.23


In [225]:
print(len(macro))
print(len(products))
print(products.index[0])
print(products.index[-1])
print(macro.index[0])
print(macro.index[-1])

140
114
2016-01-01 00:00:00
2025-06-01 00:00:00
08,2025
01,2014


In [226]:
macro = macro[::-1]
macro = macro[24:-2]
print(macro.index[0])
print(macro.index[-1])
print(products.index[0])
print(products.index[-1])

01,2016
06,2025
2016-01-01 00:00:00
2025-06-01 00:00:00


In [228]:
df = products.copy()
df[macro.columns] = macro.values

df.head()

chemical,Карбамид (FOB Южный),"Моноаммонийфосфат, MAP (FOB Балтика)",Апатитовый концетрат (FOB Morocco),Аммиак (FOB Черное море),Аммиачная селитра (FOB Черное море),Хлорид калия (CFR Ю-В Азия),Капролактам импортный контракт (Тайвань и Ю. Корея) CFR Азия,Метанол,"Бензол, CFR Япония","Этилен, CFR Китай","Инфляция - Рост индекса цен производителей (RUB, eop PPI),","Инфляция - Рост индекса потребительских цен в США, в долларах США (USD, eop CPI),","Инфляция - Рост индекса потребительских цен в США, в долларах США (USD, eop CPI), .1","Инфляция - Рост индекса цен производителей в США, в долларах США (USD, eop PPI),","Ключевая ставка, годовых","Инфляция, г/г"
Date,,,,,,,,,,,,,,,,
2016-01-01,209.875,366.25,120.0,266.666667,199.375,278.75,1153.0,185.4,556.0,987.5,-0.022,-0.1,-0.1,-0.002,11.0,9.8
2016-02-01,207.125,332.5,117.5,260.5,182.125,272.5,1163.0,150.0,546.25,927.5,-0.012,0.0,0.0,0.001,11.0,8.1
2016-03-01,199.875,345.0,117.5,267.5,180.3,267.5,1290.0,164.75,625.0,1173.75,-0.015,-0.2,-0.2,-0.002,11.0,7.3
2016-04-01,203.875,346.25,115.0,271.0,172.0,267.5,1355.0,179.9,642.0,1192.5,0.031,0.1,0.1,-0.001,11.0,7.3
2016-05-01,200.5,341.5,107.5,279.25,154.75,251.875,1278.0,201.25,631.25,1160.0,0.026,0.4,0.4,0.002,11.0,7.3


# Эксперимент

Будем пробовать предсказывать с макропоказателями и без

In [ ]:
def add_results_in_comparison_table(
    method: str, chemical: str, y_true, y_forecast
) -> pd.DataFrame:
    """Подсчёт метрик"""
    global compare_table

    result_row = {
        "method": method,
        "product": chemical,
        "MSE": mean_squared_error(y_true=y_true, y_pred=y_forecast),
        "MAE": mean_absolute_error(y_true=y_true, y_pred=y_forecast),
        "MAPE": mean_absolute_percentage_error(y_true=y_true, y_pred=y_forecast),
    }

    if compare_table is None:
        compare_table = pd.DataFrame([result_row])
    else:

        compare_table = pd.concat([compare_table, pd.DataFrame([result_row])])
        compare_table.index = np.arange(len(compare_table))
    return compare_table

In [ ]:
def evaluate_with_cv(
    data,
    chemical_list,
    method,
    method_name="aabb",
    fh=list(range(1, 13)),  # horizon
    initial_window=24,
    step_length=2,
):
    splitter = ExpandingWindowSplitter(
        initial_window=initial_window,
        step_length=step_length,
        fh=fh,
    )

    for train_idx, test_idx in tqdm(splitter.split(data)):
        train_df, test_df = data.iloc[train_idx], data.iloc[test_idx]

        forecast_df = method(train_df, fh=len(test_idx))

        for chem in chemical_list:
            y_true = test_df[chem].values
            y_pred = forecast_df[chem].values
            add_results_in_comparison_table(method_name, chem, y_true, y_pred)

    return compare_table

## Без макропоказателей

### Naive

In [232]:
def last_value_forecast(train_df, fh):
    last_vals = train_df.iloc[-1]
    forecast = pd.DataFrame(
        np.tile(last_vals.values, (fh, 1)),
        columns=train_df.columns,
        index=pd.RangeIndex(fh),
    )
    return forecast


def mean_naive_forecast(train_df, fh, window=3):
    mean_vals = train_df.iloc[-window:].mean(axis=0)
    forecast = pd.DataFrame(
        np.tile(mean_vals.values, (fh, 1)),
        columns=train_df.columns,
        index=pd.RangeIndex(fh),
    )
    return forecast

In [233]:
compare_table = None

In [234]:
compare_table = evaluate_with_cv(
    products,
    products.columns,
    method=last_value_forecast,
    method_name="last_value",
    fh=12,  # 1 year horizon
    initial_window=24,
)

compare_table = evaluate_with_cv(
    products,
    products.columns,
    method=mean_naive_forecast,
    method_name="3_mean",
    fh=12,  # 1 year horizon
    initial_window=24,
)
compare_table = evaluate_with_cv(
    products,
    products.columns,
    method=lambda x, fh: mean_naive_forecast(x, fh, 6),
    method_name="6_mean",
    fh=12,  # 1 year horizon
    initial_window=24,
)

40it [00:00, 74.02it/s]
40it [00:00, 99.50it/s]
40it [00:00, 102.79it/s]


In [235]:
print(
    "Naive forecasts, 1 year of test period, cross-validation with stride 1):\n================================================"
)
print(
    f"average (across agro) MAPE (last value) = {compare_table[compare_table["method"] == "last_value"]["MAPE"].mean() * 100:.2f} %"
)
print(
    f"average (across agro) MAPE (mean value across 3 last observations) = {compare_table[compare_table["method"] == "3_mean"]["MAPE"].mean() * 100:.2f} %"
)
print(
    f"average (across agro) MAPE (mean value across 6 last observations) = {compare_table[compare_table["method"] == "6_mean"]["MAPE"].mean() * 100:.2f} %"
)
print("================================================")

Naive forecasts, 1 year of test period, cross-validation with stride 1):
average (across agro) MAPE (last value) = 33.66 %
average (across agro) MAPE (mean value across 3 last observations) = 34.93 %
average (across agro) MAPE (mean value across 6 last observations) = 36.67 %


### Exp smoothing

In [238]:
def exp_smoothing_forecast(
    train_df, fh, trend=None, seasonal=None, seasonal_periods=None
):
    forecasts = {}
    for col in train_df.columns:
        series = pd.to_numeric(train_df[col], errors="coerce").astype(float).dropna()
        series = pd.Series(series.values)
        model = ExponentialSmoothing(
            series, trend=trend, seasonal=seasonal, seasonal_periods=seasonal_periods
        ).fit(optimized=True)
        forecasts[col] = model.forecast(fh).values
    forecast_df = pd.DataFrame(forecasts)
    forecast_df.index = pd.RangeIndex(fh)
    return forecast_df

In [241]:
compare_table = evaluate_with_cv(
    products,
    products.columns,
    method=lambda train, fh: exp_smoothing_forecast(
        train, fh, trend="add", seasonal="add", seasonal_periods=12
    ),
    method_name="exp_smoothing_greenhouse",
    initial_window=48,
)

28it [00:10,  2.78it/s]


In [242]:
print(
    f"average (across agro) MAPE (exp smoothing) = {compare_table[compare_table["method"] == "exp_smoothing_greenhouse"]["MAPE"].mean() * 100:.2f} %"
)

average (across agro) MAPE (exp smoothing) = 38.49 %


### GradBoost

In [243]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

def gbm_cross_series_forecast(train_df, fh, lags=12, n_estimators=100, max_depth=3):
    df = train_df.copy()
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.fillna(method="ffill").fillna(method="bfill")
    n = len(df)
    cols = df.columns.tolist()

    if n <= lags:
        last = df.iloc[-1].values
        return pd.DataFrame(np.tile(last, (fh, 1)), columns=cols, index=pd.RangeIndex(fh))

    X_by_h = {}
    Y_by_h = {}
    for h in range(1, fh + 1):
        X_rows = []
        Y_rows = []
        for t in range(lags, n - h):
            row = []
            for lag in range(1, lags + 1):
                row.extend(df.iloc[t - lag].values)
            if isinstance(df.index, pd.DatetimeIndex):
                row.append(df.index[t].month)
            X_rows.append(row)
            Y_rows.append(df.iloc[t + h].values)
        X_by_h[h] = np.asarray(X_rows)
        Y_by_h[h] = np.asarray(Y_rows)

    last_row = []
    for lag in range(1, lags + 1):
        last_row.extend(df.iloc[-lag].values)
    if isinstance(df.index, pd.DatetimeIndex):
        last_row.append(df.index[-1].month)
    last_row = np.asarray(last_row).reshape(1, -1)

    forecasts = np.zeros((fh, len(cols)))
    for h in range(1, fh + 1):
        X = X_by_h[h]
        Y = Y_by_h[h]
        for j, col in enumerate(cols):
            model = GradientBoostingRegressor(n_estimators=n_estimators, max_depth=max_depth)
            model.fit(X, Y[:, j])
            forecasts[h - 1, j] = model.predict(last_row)[0]

    forecast_df = pd.DataFrame(forecasts, columns=cols)
    forecast_df.index = pd.RangeIndex(fh)
    return forecast_df


In [244]:
compare_table = evaluate_with_cv(
    products,
    products.columns,
    method=lambda train, fh: gbm_cross_series_forecast(
        train, fh, lags=12, n_estimators=100, max_depth=3
    ),
    method_name="gbm_cross",
    initial_window=48,
)

0it [00:00, ?it/s]/tmp/ipykernel_135050/388658905.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill").fillna(method="bfill")
1it [00:12, 12.07s/it]/tmp/ipykernel_135050/388658905.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill").fillna(method="bfill")
2it [00:24, 12.41s/it]/tmp/ipykernel_135050/388658905.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill").fillna(method="bfill")
3it [00:38, 12.82s/it]/tmp/ipykernel_135050/388658905.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill").fillna(method="bfil

In [ ]:
print(
    f"average (across agro) MAPE (grad boost) = {compare_table[compare_table["method"] == "gbm_cross"]["MAPE"].mean() * 100:.2f} %"
)

average (across agro) MAPE (exp smoothing) = 27.93 %


## С макропоказателями